# Batch, Stochastic, and Mini-Batch Gradient Descent Lab

Explore the behavior and trade-offs of gradient descent variants: comparing epochs, updates, and seconds to converge; investigating the noise floor and decay schedules; and measuring rows seen to converge under tight versus loose tolerances.

In [ ]:
import time
import warnings
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)

## Helper Functions

Let's define standard data generators, cost, and gradient update step functions.

In [ ]:
SEED, ALPHA, TOL, UCAP = 42, 0.005, 1e-5, 1_000_000

def make(n, d, seed):
    rng = np.random.RandomState(seed)
    Z = rng.randn(n, d)
    Z = (Z - Z.mean(axis=0)) / Z.std(axis=0)
    return Z, Z @ (rng.randn(d) * 2.0) + rng.randn() * 5.0 + rng.randn(n) * 2.0

def cost(X, y, th):
    return np.sum((X @ th - y) ** 2) / len(y)          # J = (1/n)Σ(ŷ−y)²

def step(X, y, th, rows, alpha):
    g = (2 / len(rows)) * X[rows].T @ (X[rows] @ th - y[rows])
    return th - alpha * g

def label(bs, n):
    return "batch" if bs == n else "SGD (1 row)" if bs == 1 else f"mini-batch {bs}"

def train(X, y, bs, floor, alpha=ALPHA, tol=TOL, ucap=UCAP):
    """Run until the cost is within tol of floor, or until ucap updates."""
    m = len(y)
    th, rng = np.zeros(X.shape[1]), np.random.RandomState(SEED)
    epochs, updates, t0 = 0, 0, time.perf_counter()
    while True:
        epochs += 1
        order = rng.permutation(m)
        for i in range(0, m, bs):
            th = step(X, y, th, order[i:i + bs], alpha)
            updates += 1
            if updates >= ucap:
                return epochs, updates, time.perf_counter() - t0, cost(X, y, th), "update cap"
        if cost(X, y, th) - floor < tol:
            return epochs, updates, time.perf_counter() - t0, cost(X, y, th), "converged"

## 1. Rows per Update

Construct a synthetic dataset of 20,000 samples and 5 features, and find the number of updates per epoch for each batch size.

In [ ]:
n, d = 20000, 5
Z, y = make(n, d, SEED)
X = np.c_[np.ones(n), Z]
floor = cost(X, y, np.linalg.lstsq(X, y, rcond=None)[0])
print(f"1. ONE CHOICE: ROWS PER UPDATE   (n = {n}, d = {d}, alpha = {ALPHA}, tol = {TOL})")
print(f"   exact OLS minimum J_min = {floor:.6f}")
print(f"   {'variant':<16}{'rows/update':>12}{'updates/epoch':>15}")
for bs in (n, 1, 8, 32, 256, 2048):
    print(f"   {label(bs, n):<16}{bs:>12}{-(-n // bs):>15}")

## 2. Time to Converge

Compare epochs, updates, and wall-clock seconds for each variant. Watch standard full-batch and large mini-batches converge, while single-row SGD and small mini-batches fail to reach a tight tolerance due to noise.

In [ ]:
print("2. TIME TO CONVERGE   (same alpha, same start, same seed)")
print(f"   {'variant':<16}{'epochs':>8}{'updates':>10}{'seconds':>9}{'J − J_min':>13}   status")
rows = []
for bs in (n, 1, 8, 32, 256, 2048):
    ep, up, sec, J, st = train(X, y, bs, floor)
    rows.append((label(bs, n), ep, up, sec, J - floor, st))
    print(f"   {label(bs, n):<16}{ep:>8}{up:>10}{sec:>9.3f}{J - floor:>13.4e}   {st}")
ok = [r for r in rows if r[5] == "converged"]
print("   of the runs that converged, fewest epochs:  "
      + " < ".join(r[0] for r in sorted(ok, key=lambda r: r[1])))
print("   of the runs that converged, fewest seconds: "
      + " < ".join(r[0] for r in sorted(ok, key=lambda r: r[3])))
reps = [train(X, y, 256, floor)[2] for _ in range(3)]
print(f"   repeatability of the timing (mini-batch 256, 3 runs): "
      + ", ".join(f"{t:.3f}s" for t in reps))

## 3. The Noise Floor

Give each variant an equal 2-second wall-clock budget. Observe how constant-rate SGD bounces indefinitely within a noise cloud, while a decaying learning rate collapses the cloud onto the true optimum.

In [ ]:
print("3. THE NOISE FLOOR   (2 seconds each, so batch is not starved of updates)")
print(f"   {'variant':<32}{'J − J_min':>13}{'var of last 50 epochs':>24}")
for bs, decay in ((n, False), (1, False), (256, False), (1, True)):
    th, seen, rng, t0, t = np.zeros(X.shape[1]), [], np.random.RandomState(SEED), time.perf_counter(), 0
    while time.perf_counter() - t0 < 2.0:
        order = rng.permutation(n)
        for i in range(0, n, bs):
            th = step(X, y, th, order[i:i + bs], ALPHA / (1 + t / 1000.0) if decay else ALPHA)
            t += 1
        seen.append(cost(X, y, th))
    name = label(bs, n) + (" + alpha/(1+t/1000)" if decay else "")
    print(f"   {name:<32}{seen[-1] - floor:>13.4e}{np.var(seen[-50:]):>24.4e}")

## 4. Rows Seen to Reach a Loose Target

On a much larger dataset ($n=200,000$), measure how many rows must be read before the model reaches a loose target of $J_{\min} + 0.01$.

In [ ]:
NL, AL, TGT = 200000, 0.01, 0.01
ZL, yL = make(NL, d, SEED + 1)
XL = np.c_[np.ones(NL), ZL]
fl = cost(XL, yL, np.linalg.lstsq(XL, yL, rcond=None)[0])
print(f"\n4. ROWS SEEN TO REACH J_min + {TGT}   (n = {NL}, alpha = {AL}, checked every ~2000 rows)")
print(f"   {'variant':<16}{'rows seen':>13}{'x the dataset':>15}{'J − J_min':>13}   status")
for bs in (NL, 1, 32, 256):
    th, rng = np.zeros(XL.shape[1]), np.random.RandomState(SEED)
    every, seen_rows, hit, since, ups = max(1, 2000 // bs), 0, False, 0, 0
    for _ in range(50):                    # up to 50 passes, or 600k updates, whichever first
        order = rng.permutation(NL)
        for i in range(0, NL, bs):
            th = step(XL, yL, th, order[i:i + bs], AL)
            seen_rows += len(order[i:i + bs])
            since += 1
            ups += 1
            if ups >= 600_000:
                hit = False
                break
            if since >= every:
                since = 0
                if cost(XL, yL, th) - fl < TGT:
                    hit = True
                    break
        if hit or ups >= 600_000:
            break
    print(f"   {label(bs, NL):<16}{seen_rows:>13,}{seen_rows / NL:>15.2f}"
          f"{cost(XL, yL, th) - fl:>13.4e}   {'reached' if hit else 'not reached'}")

## 5. Scikit-Learn Verification

Let's compare the exact OLS minimum against scikit-learn's `LinearRegression` and `SGDRegressor`.

In [ ]:
print("5. THE SAME PROBLEM THROUGH SCIKIT-LEARN")
for name, est in (("LinearRegression", LinearRegression()),
                  ("SGDRegressor, defaults", SGDRegressor(random_state=SEED)),
                  ("SGDRegressor, constant lr",
                   SGDRegressor(max_iter=1000, eta0=0.0025, learning_rate="constant",
                                random_state=SEED, alpha=0))):
    f = est.fit(Z, y)
    ic = f.intercept_ if np.isscalar(f.intercept_) else f.intercept_[0]
    print(f"   {name:<28}J − J_min = {cost(X, y, np.r_[ic, f.coef_]) - floor:.4e}")